In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_colwidth', None)

**read the file**

In [ ]:
#read the json file
data_file_path = "../data/raw/multinli_1.0_train.jsonl"
data = pd.read_json(data_file_path, lines=True)
data.head()

**some numbers**

In [ ]:
data.shape

In [ ]:
#stat about the data
data['annotator_labels'].value_counts()

In [ ]:
#stat about the nan
data.isna().sum()

**clean the data**

In [ ]:
#select only required columns
cleaned_data = data[['annotator_labels', 'genre', 'gold_label', 'sentence1', 'sentence2']]

#remove [] from annotator_labels
cleaned_data['annotator_labels'] = cleaned_data['annotator_labels'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)
cleaned_data.head()

#make na string in sentence1 and sentence2 to actual NaN na n/a everything that looks nan
cleaned_data['sentence1'] = cleaned_data['sentence1'].replace(['na', 'n/a', 'Na', 'N/A', ''], np.nan)
cleaned_data['sentence2'] = cleaned_data['sentence2'].replace(['na', 'n/a', 'Na', 'N/A', ''], np.nan)

#elimate rows with nan in sentence1 or sentence2
cleaned_data = cleaned_data.dropna(subset=['sentence1', 'sentence2'])
cleaned_data.shape

In [ ]:
#select records where ; exist in sentence1 or sentence2
cleaned_data[cleaned_data['sentence1'].str.contains('µ') | cleaned_data['sentence2'].str.contains('µ')].shape

In [ ]:
cleaned_data.iloc[106]

**write the data to folder**

In [ ]:
#write the csv file
os.makedirs("../data/processed", exist_ok=True)
cleaned_data.to_csv("../data/processed/multinli_1.0_train_cleaned.csv", index=False, sep='µ')

## apply TF IDF

In [ ]:
#apply tf idf vectorization on sentence1 and sentence2
from sklearn.feature_extraction.text import TfidfVectorizer

# separate vectorizers to retain feature names per sentence
tfidf_vectorizer_s1 = TfidfVectorizer(max_features=100)
tfidf_vectorizer_s2 = TfidfVectorizer(max_features=100)

tfidf_sentence1 = tfidf_vectorizer_s1.fit_transform(data['sentence1']).toarray()
tfidf_sentence2 = tfidf_vectorizer_s2.fit_transform(data['sentence2']).toarray()

In [ ]:
#print the first 5 rows of the tfidf vectors
print(tfidf_sentence1[0])

In [ ]:
#use the tfidf vectors as features (concatenate sentence1 and sentence2)
feature_names_s1 = [f"s1_{w}" for w in tfidf_vectorizer_s1.get_feature_names_out()]
feature_names_s2 = [f"s2_{w}" for w in tfidf_vectorizer_s2.get_feature_names_out()]
all_feature_names = feature_names_s1 + feature_names_s2

features = pd.DataFrame(
    np.hstack([tfidf_sentence1, tfidf_sentence2]),
    columns=all_feature_names
)
features.head()

#add the features to the cleaned_data
cleaned_data = pd.concat([cleaned_data.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

#drop sentence1 and sentence2
cleaned_data = cleaned_data.drop(columns=['sentence1', 'sentence2'])
cleaned_data.head()

## split to train and dev

In [ ]:
#split the data into train and test
from sklearn.model_selection import train_test_split

#drop columns annotator_labels gold_label
cleaned_data_train = cleaned_data.drop(columns=['annotator_labels', 'gold_label'])

X_train, X_test, y_train, y_test = train_test_split(cleaned_data_train, cleaned_data['gold_label'], test_size=0.2, random_state=42)

## turn labels to one hot encoding

In [ ]:
# one-hot encode genre; keep labels as 1D series
X_train = pd.get_dummies(X_train, columns=['genre'])
X_test = pd.get_dummies(X_test, columns=['genre'])

# align columns between train and test after encoding
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## train an SVM for classification

In [ ]:
# train a random forest classifier
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.fit(X_train, y_train)

## evaluate the model

In [ ]:
#calculate the precision on the test dataset
from sklearn.metrics import precision_score

y_pred = clf.predict(X_test)
precision = precision_score(y_test, y_pred, average='macro')
print(f'Precision: {precision}')